# Chronos-X: Cloud Batch Predictor 🚀

Use this notebook to run the heavy AI forecasting job on Google Colab's free T4 GPU.

### Steps:
1. **Enable GPU**: Go to `Runtime` > `Change runtime type` > Select `T4 GPU`
2. **Upload Data**: Open the Files sidebar (folder icon) and drag-and-drop your local `pulse_vectors.parquet` file.
3. **Run All**: Click `Runtime` > `Run all`.
4. **Download**: When finished, download the generated `predictions.parquet` file.

## 1. Install Dependencies

In [1]:
!pip install polars numpy torch accelerate transformers git+https://github.com/amazon-science/chronos-forecasting.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/amazon-science/chronos-forecasting.git to c:\users\prana\appdata\local\temp\pip-req-build-xlw0ogd4
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit f889ae66477b53f6beb130f5c7b13590b29a1035
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git 'C:\Users\prana\AppData\Local\Temp\pip-req-build-xlw0ogd4'

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


## 2. define AI Forecaster Code
*(Pasted from our local `src/ai_forecaster.py`)*

In [2]:
import polars as pl
import numpy as np
from typing import Optional, Union
from pathlib import Path
from datetime import date, datetime
import torch
from chronos import ChronosPipeline
import json
import time

def _parse_date(date_input: Union[str, date]) -> date:
    if isinstance(date_input, date):
        return date_input
    return datetime.strptime(date_input, "%Y-%m-%d").date()

def compute_cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / (norm1 * norm2))

class AIForecaster:
    def __init__(self, model_name: str = "amazon/chronos-t5-base"):
        self.model_name = model_name
        self._pipeline = None
    
    @property
    def pipeline(self):
        if self._pipeline is None:
            print(f"Loading Chronos model: {self.model_name}...")
            # Use float16 for GPU speedup if available
            torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
            device_map = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"Using device: {device_map}")
            
            self._pipeline = ChronosPipeline.from_pretrained(
                self.model_name,
                device_map=device_map,
                torch_dtype=torch_dtype,
            )
            print("Model loaded successfully!")
        return self._pipeline
    
    def forecast_pin(self, pin_history: list[float], horizon: int = 6, num_samples: int = 20) -> dict:
        # Convert to tensor
        context = torch.tensor(pin_history, dtype=torch.float32)
        
        # Generate probabilistic forecast (positional args per official API)
        # Returns shape [num_series, num_samples, prediction_length]
        forecast = self.pipeline.predict(context, horizon, num_samples=num_samples)
        
        # Extract quantiles for uncertainty bounds (using numpy)
        forecast_np = forecast[0].numpy()  # [num_samples, prediction_length]
        low, median, high = np.quantile(forecast_np, [0.1, 0.5, 0.9], axis=0)
        
        return {
            "median": median.tolist(),
            "lower_bound": low.tolist(),
            "upper_bound": high.tolist()
        }
class HybridPredictor:
    def __init__(self, model_name: str = "amazon/chronos-t5-base"):
        self.ai_forecaster = AIForecaster(model_name)
        self.pulse_table = None
        
    def load_pulse_table(self, df: pl.DataFrame):
        self.pulse_table = df
        return self
    
    def predict_hybrid(self, pincode, ref_month, target_month, k=5, ai_weight=0.5):
        # Re-implemented simplified version for Colab single-file execution
        ref_date = _parse_date(ref_month)
        
        # 1. Get Current
        row = self.pulse_table.filter(
            (pl.col("pincode") == pincode) & (pl.col("month") == ref_date)
        )
        if len(row) == 0: return None
        
        target_vector = np.array([
            row["growth_norm"][0], row["flow_norm"][0], row["labor_norm"][0]
        ])
        state = row["state"][0]
        
        # 2. Spatial Search
        # (For speed in Colab, we'll do a focused search or assume pre-computed parallels if needed)
        # Here we do full search - GPU should handle AI part fast, CPU handles this part ok
        candidates = self.pulse_table.filter(
            (pl.col("month") == ref_date) & 
            (pl.col("pincode") != pincode) & 
            (pl.col("state") != state)
        )
        
        # Simple loop for brevity in Colab (can be optimized)
        best_parallels = []
        # Optimizing: Calculate on top 1000 random candidates to save time if needed
        # Or use vectorized if dataframe is small enough. For 19k rows, vector is fine.
        matrix = candidates.select(["growth_norm", "flow_norm", "labor_norm"]).to_numpy()
        if len(matrix) == 0: return None
        
        # Cosine sim with broadcasting
        target_norm = target_vector / (np.linalg.norm(target_vector) + 1e-9)
        matrix_norm = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
        sims = np.dot(matrix_norm, target_norm)
        
        top_k_idx = np.argsort(sims)[-k:][::-1]
        
        parallels = []
        weighted_sum = np.zeros(3)
        total_weight = 0
        
        cand_pincodes = candidates["pincode"].to_numpy()
        cand_states = candidates["state"].to_numpy()
        
        # Get future outcomes for these parallels
        future_date = _parse_date(target_month)
        
        for idx in top_k_idx:
            cand_pin = cand_pincodes[idx]
            sim = sims[idx]
            
            f_row = self.pulse_table.filter(
                (pl.col("pincode") == cand_pin) & (pl.col("month") == future_date)
            )
            
            if len(f_row) > 0:
                outcome = np.array([
                    f_row["growth_norm"][0], f_row["flow_norm"][0], f_row["labor_norm"][0]
                ])
                weighted_sum += outcome * sim
                total_weight += sim
                parallels.append({"pincode": cand_pin, "state": cand_states[idx]})
        
        if total_weight > 0:
            spatial_pred = weighted_sum / total_weight
        else:
            spatial_pred = target_vector
            
        # 3. AI Forecast (Chronos)
        # Get history for this PIN
        pin_history = self.pulse_table.filter(pl.col("pincode") == pincode).sort("month")
        
        ai_pred = np.zeros(3)
        ai_forecasts = {}
        
        for i, col in enumerate(["pulse_growth", "pulse_flow", "pulse_labor"]):
             # Use pulse_* raw or *_norm? Forecaster used *_norm in script update
             col_norm = ["growth_norm", "flow_norm", "labor_norm"][i]
             hist = pin_history[col_norm].to_list()
             
             # Predict next step + horizon
             f = self.ai_forecaster.forecast_pin(hist, horizon=6)
             ai_pred[i] = f["median"][0] # Next month
             ai_forecasts[col_norm] = f
             
        # 4. Ensemble
        ensemble = (spatial_pred * (1-ai_weight)) + (ai_pred * ai_weight)
        
        return {
            "ensemble": ensemble,
            "current": target_vector,
            "parallels": parallels,
            "ai_forecasts": ai_forecasts,
            "confidence": float(np.mean(sims[top_k_idx]))
        }

ModuleNotFoundError: No module named 'polars'

## 3. Run Batch Prediction

In [ ]:
def run_batch():
    if not Path("pulse_vectors.parquet").exists():
        print("❌ ERROR: Please upload 'pulse_vectors.parquet' to the files sidebar!")
        return

    print("Loading data...")
    try:
        df = pl.read_parquet("pulse_vectors.parquet")
    except Exception as e:
        print(f"❌ ERROR loading parquet: {e}")
        return
        
    predictor = HybridPredictor()
    predictor.load_pulse_table(df)
    
    # Get target PINs (Nov 2025)
    ref_date = date(2025, 11, 1)
    pins = df.filter(pl.col("month") == ref_date)["pincode"].unique().to_list()
    
    print(f"Starting processing for {len(pins)} PINs on GPU...")
    print("(This typically takes ~45 mins on T4 GPU)")
    
    results = []
    start = time.time()
    error_count = 0
    
    for idx, pin in enumerate(pins):
        if idx % 100 == 0:
            elapsed = time.time() - start
            print(f"Processed {idx}/{len(pins)} ({elapsed:.1f}s) | Errors: {error_count}")
            
        try:
            out = predictor.predict_hybrid(pin, "2025-11-01", "2025-12-01")
            if out:
                res = {
                    "pincode": pin,
                    "current_growth": out["current"][0],
                    "current_flow": out["current"][1],
                    "current_labor": out["current"][2],
                    "next_growth": out["ensemble"][0],
                    "next_flow": out["ensemble"][1],
                    "next_labor": out["ensemble"][2],
                    "forecast_growth": json.dumps(out["ai_forecasts"]["growth_norm"]["median"]),
                    "forecast_flow": json.dumps(out["ai_forecasts"]["flow_norm"]["median"]),
                    "forecast_labor": json.dumps(out["ai_forecasts"]["labor_norm"]["median"]),
                    "confidence": out["confidence"],
                    "parallel_states": ", ".join([p["state"] for p in out["parallels"]][:3])
                }
                results.append(res)
        except Exception as e:
            error_count += 1
            if error_count <= 5:
                print(f"❌ Error for PIN {pin}: {e}")
                import traceback
                traceback.print_exc()
            pass
            
    # Save
    if len(results) > 0:
        pl.DataFrame(results).write_parquet("predictions.parquet")
        print("✅ Done! Download 'predictions.parquet' from the sidebar.")
    else:
        print("❌ NO RESULTS GENERATED.")

run_batch()